# Tech Challenge Fase 1 — Diagnóstico de Diabetes
## 4. Interpretabilidade dos Modelos
Este notebook apresenta técnicas de interpretabilidade para entender quais features
mais influenciam as predições dos modelos:
- **Feature Importance** (Random Forest)
- **SHAP** (SHapley Additive exPlanations) — aplicado à Regressão Logística (modelo vencedor)

## 4.1 Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import shap

sns.set_theme(style='whitegrid')
print('Bibliotecas importadas com sucesso!')
print(f'SHAP versão: {shap.__version__}')

## 4.2 Carregamento dos dados e modelos

In [ ]:
# Dados
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').values.ravel()

# Corrigir NaN remanescentes
for col in X_train.columns:
    mediana = X_train[col].median()
    X_train[col] = X_train[col].fillna(mediana)
    X_test[col]  = X_test[col].fillna(mediana)

# Modelos
with open('../data/models/logistic_regression.pkl', 'rb') as f:
    lr = pickle.load(f)

with open('../data/models/random_forest.pkl', 'rb') as f:
    rf = pickle.load(f)

feature_names = list(X_train.columns)
print(f'Dados carregados: {X_train.shape}')
print(f'Features: {feature_names}')

## 4.3 Feature Importance — Random Forest

O Random Forest calcula a importância de cada feature com base em quanto ela
reduz a impureza (Gini) nas árvores de decisão.

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

fi_df = pd.DataFrame({
    'Feature': [feature_names[i] for i in indices],
    'Importância': importances[indices]
})

print('=== FEATURE IMPORTANCE — RANDOM FOREST ===')
print(fi_df.to_string(index=False))

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(fi_df['Feature'][::-1], fi_df['Importância'][::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Importância (Gini)', fontsize=12)
ax.set_title('Feature Importance — Random Forest', fontsize=14)
for bar, val in zip(bars, fi_df['Importância'][::-1]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../notebooks/feature_importance_rf.png', dpi=150)
plt.show()

## 4.4 Coeficientes da Regressão Logística

Na Regressão Logística, os coeficientes indicam o peso de cada feature na predição.
Valores positivos aumentam a probabilidade de diabetes; negativos, diminuem.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coeficiente': lr.coef_[0]
}).sort_values('Coeficiente', ascending=False)

print('=== COEFICIENTES — REGRESSÃO LOGÍSTICA ===')
print(coef_df.to_string(index=False))

# Gráfico
cores = ['tomato' if c > 0 else 'steelblue' for c in coef_df['Coeficiente']]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df['Feature'], coef_df['Coeficiente'], color=cores, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coeficiente', fontsize=12)
ax.set_title('Coeficientes — Regressão Logística\n(vermelho = aumenta risco | azul = reduz risco)', fontsize=13)
plt.tight_layout()
plt.savefig('../notebooks/coeficientes_lr.png', dpi=150)
plt.show()

## 4.5 SHAP — Regressão Logística (modelo vencedor)

SHAP (SHapley Additive exPlanations) explica a contribuição de cada feature
para cada predição individual, com base na teoria dos jogos cooperativos.

É mais robusto que feature importance pois considera interações entre features.

In [ ]:
# Criar explainer SHAP para Regressão Logística
explainer = shap.LinearExplainer(lr, X_train, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_test)

print(f'SHAP values calculados para {X_test.shape[0]} amostras de teste')
print(f'Shape dos SHAP values: {shap_values.shape}')

## 4.6 SHAP — Summary Plot (impacto global de cada feature)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Summary Plot — Regressão Logística', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 SHAP — Bar Plot (importância média global)

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('SHAP — Importância Média por Feature', fontsize=14)
plt.tight_layout()
plt.savefig('../notebooks/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.8 SHAP — Waterfall Plot (explicação de uma predição individual)

Mostra como cada feature contribuiu para a predição de um paciente específico.

In [ ]:
# Escolher um paciente com diabetes real (y_test == 1)
idx_diabetico = np.where(y_test == 1)[0][0]

print(f'Paciente analisado: índice {idx_diabetico}')
print(f'Diagnóstico real: {"Com Diabetes" if y_test[idx_diabetico] == 1 else "Sem Diabetes"}')
print(f'Predição do modelo: {"Com Diabetes" if lr.predict(X_test.iloc[[idx_diabetico]])[0] == 1 else "Sem Diabetes"}')
print(f'Probabilidade de diabetes: {lr.predict_proba(X_test.iloc[[idx_diabetico]])[0][1]:.2%}')

# Waterfall
shap_explanation = shap.Explanation(
    values=shap_values[idx_diabetico],
    base_values=explainer.expected_value,
    data=X_test.iloc[idx_diabetico].values,
    feature_names=feature_names
)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, show=False)
plt.title(f'SHAP Waterfall — Paciente {idx_diabetico} (Com Diabetes)', fontsize=13)
plt.tight_layout()
plt.savefig('../notebooks/shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.9 Comparação: Feature Importance vs SHAP

In [ ]:
# SHAP mean absolute values
shap_importancia = pd.DataFrame({
    'Feature': feature_names,
    'SHAP (|médio|)': np.abs(shap_values).mean(axis=0),
    'RF Importance': [importances[feature_names.index(f)] for f in feature_names]
}).sort_values('SHAP (|médio|)', ascending=False)

print('=== COMPARAÇÃO: SHAP vs FEATURE IMPORTANCE ===')
print(shap_importancia.round(4).to_string(index=False))

## 4.10 Conclusões da Interpretabilidade

**Achados principais:**

1. **Glucose** é consistentemente a feature mais importante em ambos os métodos — alto nível de glicose é o principal preditor de diabetes, alinhado com o conhecimento clínico
2. **BMI** e **Age** aparecem como segundo e terceiro fatores mais relevantes
3. **BloodPressure** e **SkinThickness** têm baixa importância — podem ter pouco poder preditivo neste dataset
4. O SHAP Waterfall mostra como interpretar a predição de um paciente individual, tornando o modelo auditável por profissionais de saúde

**Implicação clínica:**

O modelo pode ser usado como ferramenta de triagem, priorizando pacientes com glicose elevada, IMC alto e idade avançada para avaliação médica detalhada. A transparência dos valores SHAP permite que o(a) médico(a) entenda e questione cada predição, mantendo o controle clínico sobre o diagnóstico final.